# 01. Min cost

**Tier:** Deep. **Why you'd use this:** You need a pattern that turns a nested scan into one clear pass.

**After this lesson you can:** store the best answer for each prefix; build the answer from smaller subproblems.
About 35 minutes. **Uses:** [Loops](../../course/03-control-flow/02-loops-lesson.ipynb), [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Memoization](../../course/09-recursion/04-memoization-lesson.ipynb)


## Quick review

1. What makes a problem a fit for DP?
2. What does each DP cell store in a min-cost prefix problem?

<details><summary>Answers</summary>

For 01 min cost lesson: skim [Loops](../../course/03-control-flow/02-loops-lesson.ipynb), [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Memoization](../../course/09-recursion/04-memoization-lesson.ipynb) if needed.
</details>


## Recognize it

`cost[i]` is what you pay to step off stair `i`. You may start on stair 0 or stair 1, climb 1 or 2 stairs at a time, and you want the cheapest way to reach the top, just past the last stair.

Signals: "minimum", "maximum", or "number of ways"; a choice at each step; the best answer built from the best answers to smaller versions. A look-alike that is not this pattern: if one locally best choice is always safe, it is greedy.

Marker for uniqueness: 13-dynamic-programming:01-min-cost-lesson.


## Why it works this way

A stair's cost is what you pay to step off it. You may start on stair 0 or stair 1, climb one or two stairs at a time, and stop when you are just past the last stair. Call that place the top.

Let `reach[i]` be the cheapest cost to stand on stair `i`, before you pay to leave it. You arrive from stair `i - 1` by paying `cost[i - 1]`, or from stair `i - 2` by paying `cost[i - 2]`. The cheaper of those two arrivals is `reach[i]`. Standing on stair 0 or stair 1 is free, because the problem lets you start on either one. The top is index `len(cost)`, one past the last stair. You do not pay a cost for the top. You only pay for a stair you step off.

Asking the question recursively, with no memory, repeats itself. From stair `i` you ask about `i + 1` and `i + 2`, and each of those asks the same later stairs again. The number of calls grows exponentially: ten stairs is a few hundred calls, and twenty-five is hundreds of thousands. Those repeated questions are overlapping subproblems. Once you know the cheapest way to stand on a stair, that answer never changes. Store it, and each stair is solved once.

You do not need the whole list. Each stair uses only the two answers before it, so two variables are enough. One pass from the bottom to the top is then proportional to the number of stairs in time, and the extra memory is a constant.

A greedy choice, "leave the cheaper of the next two stairs and never look back", is a different rule. It can walk into an expensive region that a slightly worse first step would have avoided. The recurrence keeps the cheaper whole path, not the cheaper neighbor. That is why both arrivals are considered at every stair.

The stored answers are the same algorithm in two orders. The loop below fills them from the bottom. `functools.cache` fills them on demand, the first time each stair is asked. Both refuse to solve a stair twice.

Marker for uniqueness: 13-dynamic-programming:01-min-cost-lesson.


## When to use it

Use this when the problem asks for a minimum, a maximum, or a number of ways, and each answer is built from smaller answers of the same kind. The choices here are "climb one" and "climb two". Use greedy instead when one local choice is always safe, such as taking the largest coin when every coin divides the next. Counting climbs, with no costs, is this recurrence without the `cost` array. A single contiguous span with a running condition is a sliding window, not this pattern.

Marker for uniqueness: 13-dynamic-programming:01-min-cost-lesson.


## Brute force, and why it's slow

Count the calls. `best(i)` is the cheapest total from standing on stair `i` to the top.


In [1]:
def min_cost_slow(cost):
    calls = 0

    def best(i):  # cheapest total from standing on stair i to the top
        nonlocal calls
        calls += 1
        if i >= len(cost):
            return 0
        return cost[i] + min(best(i + 1), best(i + 2))

    answer = min(best(0), best(1))
    return answer, calls


for n in (10, 20, 25):
    print(n, min_cost_slow([1] * n))


10 (5, 464)
20 (10, 57312)
25 (12, 635620)


## Key insight

Let `reach[i]` be the cheapest cost to stand on stair `i`. You arrive from `i - 1` (paying `cost[i - 1]`) or from `i - 2` (paying `cost[i - 2]`), so `reach[i] = min(reach[i - 1] + cost[i - 1], reach[i - 2] + cost[i - 2])`. Each value needs only the two before it.


## Worked example

Min cost climbing stairs. Pay `cost[i]` when you step off stair `i`. Reach the top, one past the last stair, as cheaply as you can.


In [2]:
def min_cost(cost: list[int]) -> int:
    # 1. State: the cheapest way to stand on the previous two stairs.
    # 2. Base: standing on stair 0 or 1 is free.
    two_back, one_back = 0, 0
    # 3. Transition: arrive from one below or two below, paying for the stair you leave.
    for i in range(2, len(cost) + 1):
        two_back, one_back = one_back, min(one_back + cost[i - 1], two_back + cost[i - 2])
    # 4. The top is stair len(cost), one past the last.
    return one_back


print(min_cost([10, 15, 20]))
print(min_cost([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]))


15
6


## Trace

`[10, 15, 20]`:

| i | from i−1 | from i−2 | reach[i] |
|---|---|---|---|
| 0 | — | — | 0 |
| 1 | — | — | 0 |
| 2 | 0 + 15 | 0 + 10 | 10 |
| 3 (top) | 10 + 20 | 0 + 15 | **15** |


## Cost

One pass with two variables: time grows with the number of stairs, and extra memory is two numbers. That beats the exponential walk because each stair is solved once.


## Variants

Not solved here: [Climbing Stairs](../../challenges/problems/17-climbing-stairs.ipynb), [House Robber](../../challenges/problems/18-house-robber.ipynb), [Coin Change](../../challenges/problems/19-coin-change.ipynb), [Unique Paths](../../challenges/problems/37-unique-paths.ipynb).


## Common mistake

This bug is silent. The loop stops one stair early and returns the cost of standing on the last stair, not the top. There is no traceback. The print is `10`, and the answer is `15`.


In [3]:
def min_cost_wrong(cost):
    reach = [0, 0]
    for i in range(2, len(cost)):  # stops one short of the top
        reach.append(min(reach[i - 1] + cost[i - 1], reach[i - 2] + cost[i - 2]))
    return reach[-1]


print(min_cost_wrong([10, 15, 20]))


10


The fix includes the top. `range` must stop after `len(cost)`, which is one past the last stair. The buggy cell prints `10` for `[10, 15, 20]` because the loop never reaches the top landing.


In [4]:
print(min_cost([10, 15, 20]))


15


## Python idioms

Tuple assignment rolls the two variables forward. `functools.cache` turns `best(i)` into the memoized version in one line. `cache_info` shows how many calls were stored and how many were reused.


In [5]:
import functools


def min_cost_cached(cost: list[int]):
    @functools.cache
    def best(i: int) -> int:
        if i >= len(cost):
            return 0
        return cost[i] + min(best(i + 1), best(i + 2))

    answer = min(best(0), best(1))
    return answer, best.cache_info()


print(min_cost_cached([10, 15, 20]))


(15, CacheInfo(hits=3, misses=5, maxsize=None, currsize=5))


## Recap

- The cheapest way to stand on a stair is the cheaper of the two ways to arrive.
- Store those answers, or the call tree repeats them.
- The top is one past the last stair, and you do not pay a cost for it.

Next: [practice](01-min-cost-practice.ipynb).
